In [1]:
# from sam2.build_sam import build_sam2
# from sam2.sam2_image_predictor import SAM2ImagePredictor
# import torch
from nd2reader import ND2Reader
import tifffile
import numpy as np
from matplotlib import pyplot as plt
import glob
import sys
import numpy as np
import os

os.chdir('/Users/bebr1814/projects/anabaena/scratch_data/playground')

In [ ]:
data_path = '/Users/bebr1814/projects/anabaena/scratch_data/20241010_ZMB_Anabaena/20241010_001_ZMB/'
# nd2_files = ['/Users/bebr1814/projects/anabaena/scratch_data/2020.3.5_ana33047_minusn_0003.nd2']

In [ ]:
nd2_files = glob.glob(os.path.join(data_path, '*.nd2'))
nd2_files

In [ ]:
def normalize_image(image):
    # Normalize to the range [0, 1] for floats
    image_min = image.min()
    image_max = image.max()
    return (image - image_min) / (image_max - image_min)

def read_nd2_images(nd2_file):
    nd2 = ND2Reader(nd2_file)
    print(nd2_file)
    print(nd2.metadata)

    nd2.iter_axes = 'v'  # Iterate over frames by 'v'
    
    for v in range(nd2.sizes['v']):
        # Set the iterating axis to 'c' for channels
        nd2.iter_axes = 'c'

        # Get the brightfield and the two fluorescence channels
        brightfield = nd2.get_frame_2D(c=0, v=v)  # Brightfield (grayscale)
        cy5 = nd2.get_frame_2D(c=1, v=v)  # Cy5 (fluorescence)
        texasred = nd2.get_frame_2D(c=2, v=v)  # Texas Red (fluorescence)

        yield (brightfield, cy5, texasred)


In [ ]:

# Visualization
fig, axes = plt.subplots(5, 3, figsize=(12,20))
iax = axes.flatten()

for i, (brightfield, cy5, texasred) in enumerate(read_nd2_images(nd2_files[0])):
    if i == 5:  # Show only 5 sets of images
        break

    brightfield_norm = normalize_image(brightfield)
    cy5_norm = normalize_image(cy5)
    texasred_norm = normalize_image(texasred)
    # brightfield_norm = brightfield
    # cy5_norm = cy5
    # texasred_norm = texasred

    iax[i * 3].imshow(brightfield_norm, cmap='gray')
    iax[i * 3].axis('off')
    
    iax[i * 3 + 1].imshow(cy5_norm, cmap='Reds_r')
    iax[i * 3 + 1].axis('off')
    
    iax[i * 3 + 2].imshow(texasred_norm, cmap='Reds_r')
    iax[i * 3 + 2].axis('off')


plt.show()



### Convert all to Tiff

In [2]:
sys.path.append('/Users/bebr1814/projects/anabaena/src')
from file_formatting.standardize_file_types import ImageHandler as ImageHandler

In [3]:
image = ImageHandler('/Users/bebr1814/projects/anabaena/scratch_data/20241010_ZMB_Anabaena/20241010_001_ZMB/01.nd2')

In [4]:
image.read_image()

Successfully read ND2 file: /Users/bebr1814/projects/anabaena/scratch_data/20241010_ZMB_Anabaena/20241010_001_ZMB/01.nd2


array([[[24911, 25731, 25575, ..., 25739, 25773, 25016],
        [24614, 26062, 26073, ..., 26096, 25792, 24654],
        [24704, 26172, 26084, ..., 25919, 25941, 24896],
        ...,
        [24925, 26612, 26106, ..., 27655, 26772, 25900],
        [25136, 26239, 26010, ..., 26952, 26781, 25457],
        [25193, 25879, 26427, ..., 27290, 26931, 25597]],

       [[  242,   268,   290, ...,   318,   308,   309],
        [  282,   270,   278, ...,   297,   287,   271],
        [  252,   254,   272, ...,   309,   272,   300],
        ...,
        [  338,   295,   303, ...,   774,   713,   647],
        [  297,   322,   360, ...,   719,   667,   611],
        [  329,   351,   368, ...,   656,   722,   573]],

       [[  151,   160,   163, ...,   186,   177,   158],
        [  142,   154,   164, ...,   180,   186,   149],
        [  146,   165,   194, ...,   177,   188,   180],
        ...,
        [  221,   166,   179, ...,   328,   292,   332],
        [  185,   178,   182, ...,   328,   2

In [5]:
image.save_image('/Users/bebr1814/projects/anabaena/scratch_data/20241010_ZMB_Anabaena/20241010_001_ZMB/01.tiff')

Successfully saved image using tifffile: /Users/bebr1814/projects/anabaena/scratch_data/20241010_ZMB_Anabaena/20241010_001_ZMB/01.tiff


In [6]:
image.image.shape

(3, 2044, 2048)

In [7]:
image.image[0].shape

(2044, 2048)

In [ ]:

# Handle saving as TIFF
if file_extension in ['.tif', '.tiff']:
	# Try saving TIFF using Pillow first
	# try:
	#     if isinstance(self.image, Image.Image):
	#         self.image.save(save_path)
	#         print(f"Successfully saved image using Pillow: {save_path}")
	#     else:
	#         raise ValueError("Image not in Pillow format, trying tifffile")
	# except Exception as pillow_save_error:
		# print(f"Error saving with Pillow, trying tifffile: {pillow_save_error}")
	
	# Try saving TIFF using tifffile if Pillow fails
	try:
		# tiff.imwrite(save_path, self.image)
		# We need to write each channel separately
		n_channels = self.image.shape[0]
		for c in range(n_channels):
			tiff.imwrite(save_path.replace('.tif',f'.channel_{c}.tif'), self.image[c])

		print(f"Successfully saved image using tifffile: {save_path}")
	except Exception as tifffile_save_error:
		print(f"Error saving image with tifffile: {tifffile_save_error}")


In [ ]:
python ImageHandler.py --input_file